In [ ]:
!pip install transformers -U
!pip install gensim
import pandas as pd
data = pd.read_csv("/content/dataset.csv", engine="python")
data.head()

In [ ]:
data['text'] = data['text'].astype(str).fillna("")
data= data[data['text'].str.strip() != ""]
label_map = {"Positive": 0, "Negative": 1, "Neutral": 2}
data['decision'] = data['decision'].map(label_map)

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
import torch
from transformers import TrainingArguments, Trainer
from transformers import BertTokenizer, BertForSequenceClassification
# Load BERT tokenizer and model
tokenizer2 = BertTokenizer.from_pretrained('bert-base-uncased')
model2 = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=3)

In [ ]:
X = list(data["text"])
y = list(data["decision"])
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2,stratify=y)


In [ ]:
from gensim.models import Word2Vec
sentences = [text.split() for text in X_train]
w2v_model = Word2Vec(
    sentences,
    vector_size=300,
    window=5,
    min_count=2,
    workers=4,
    epochs=10
)

In [ ]:
import numpy as np

def sentence_w2v(sentence, model, dim=300):
    words = sentence.split()
    vectors = [model.wv[w] for w in words if w in model.wv]
    if len(vectors) == 0:
        return np.zeros(dim)
    return np.mean(vectors, axis=0)

In [ ]:
X_train_w2v = np.array([sentence_w2v(s, w2v_model) for s in X_train])
X_val_w2v   = np.array([sentence_w2v(s, w2v_model) for s in X_val])

In [ ]:
class FusionDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, w2v_embeddings, labels):
        self.encodings = encodings
        self.w2v = torch.tensor(w2v_embeddings, dtype=torch.float)
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["w2v"] = self.w2v[idx]
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
X_train_tokenized = tokenizer2(X_train, padding=True, truncation=True, max_length=512)
X_val_tokenized = tokenizer2(X_val, padding=True, truncation=True, max_length=512)

In [ ]:
train_dataset = FusionDataset(X_train_tokenized, X_train_w2v, y_train)
val_dataset   = FusionDataset(X_val_tokenized, X_val_w2v, y_val)

In [ ]:
import torch.nn as nn
from transformers import BertModel
class BertW2VEarlyFusion(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.bert = BertModel(config)
        self.dropout = nn.Dropout(0.3);

        self.classifier = nn.Linear(
            config.hidden_size + 300,   # 768 + 300
            config.num_labels
        )

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        w2v=None,
        labels=None
    ):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_embedding = outputs.last_hidden_state[:, 0, :]  # [CLS]
        fused = torch.cat((cls_embedding, w2v), dim=1)
        fused = self.dropout(fused)

        logits = self.classifier(fused)

        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)

        return {"loss": loss, "logits": logits}

In [ ]:
from transformers import BertConfig
from transformers import BertModel # Only need BertModel

config = BertConfig.from_pretrained(
    "bert-base-uncased",
    num_labels=3
)

# Instantiate the custom model directly
model = BertW2VEarlyFusion(config)
bert_base_model = BertModel.from_pretrained("bert-base-uncased")
model.bert.load_state_dict(bert_base_model.state_dict())

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    # Convert labels to one-hot encoding for AUC calculation
    labels_one_hot = np.eye(len(np.unique(labels)))[labels]
    metrics = {
        "accuracy": accuracy_score(labels, preds),
        "precision_weighted": precision_score(labels, preds, average="weighted"),
        "recall_weighted": recall_score(labels, preds, average="weighted"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
        "precision_macro": precision_score(labels, preds, average="macro"),
        "recall_macro": recall_score(labels, preds, average="macro"),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "precision_micro": precision_score(labels, preds, average="micro"),
        "recall_micro": recall_score(labels, preds, average="micro"),
        "f1_micro": f1_score(labels, preds, average="micro"),
    }

    # Add AUC only if there are enough classes and probabilities are available
    try:
        if logits.shape[1] > 2:
            # Multiclass AUC (One-vs-Rest)
            auc_score = roc_auc_score(labels_one_hot, logits, average='weighted', multi_class='ovr')
        else:
            # Binary AUC
            auc_score = roc_auc_score(labels, logits[:, 1]) # Assuming column 1 is positive class score
        metrics["auc"] = auc_score
    except ValueError as e:
        print(f"Could not compute AUC: {e}")
        # Handle cases where AUC cannot be computed (e.g., all labels are the same)

    return metrics

args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()
trainer.evaluate()

In [ ]:
all_sentences = [text.split() for text in X]
w2v_full_corpus = Word2Vec(
    all_sentences,
    vector_size=300,
    window=5,
    min_count=2,
    workers=4,
    epochs=10
)
print("Word2Vec model trained on the full dataset (w2v_full_corpus).")

In [ ]:
from sklearn.model_selection import StratifiedKFold

all_fold_metrics = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_index, val_index) in enumerate(skf.split(X, y)):
    print(f"\n--- Fold {fold+1}/{skf.n_splits} ---")

    # a. Split X and y into train_index and val_index using the current fold's indices.
    X_train_fold = [X[i] for i in train_index]
    X_val_fold = [X[i] for i in val_index]
    y_train_fold = [y[i] for i in train_index]
    y_val_fold = [y[i] for i in val_index]

    # c. Generate Word2Vec embeddings for X_train_fold and X_val_fold
    X_train_w2v_fold = np.array([sentence_w2v(s, w2v_full_corpus) for s in X_train_fold])
    X_val_w2v_fold = np.array([sentence_w2v(s, w2v_full_corpus) for s in X_val_fold])

    # d. Tokenize X_train_fold and X_val_fold using tokenizer2
    X_train_tokenized_fold = tokenizer2(X_train_fold, padding=True, truncation=True, max_length=512)
    X_val_tokenized_fold = tokenizer2(X_val_fold, padding=True, truncation=True, max_length=512)

    # e. Create FusionDataset instances for the current train and validation sets
    train_dataset_fold = FusionDataset(X_train_tokenized_fold, X_train_w2v_fold, y_train_fold)
    val_dataset_fold = FusionDataset(X_val_tokenized_fold, X_val_w2v_fold, y_val_fold)

    # f. Re-initialize a new instance of BertW2VEarlyFusion and load pre-trained weights
    model_fold = BertW2VEarlyFusion(config)
    bert_base_model_for_fold = BertModel.from_pretrained("bert-base-uncased")
    model_fold.bert.load_state_dict(bert_base_model_for_fold.state_dict())

    # g. Create a new Trainer instance for the current fold
    # Re-using the 'args' and 'compute_metrics' defined previously
    trainer_fold = Trainer(
        model=model_fold,
        args=args,
        train_dataset=train_dataset_fold,
        eval_dataset=val_dataset_fold,
        compute_metrics=compute_metrics
    )

    # h. Train the model
    trainer_fold.train()

    # i. Evaluate the trained model and store metrics
    fold_metrics = trainer_fold.evaluate()
    all_fold_metrics.append(fold_metrics)

print("Cross-validation complete. All fold metrics collected.")

In [ ]:
aggregated_metrics = {}

# Aggregate metrics from all folds
for fold_metrics in all_fold_metrics:
    for metric_name, metric_value in fold_metrics.items():
        # Exclude non-numeric metrics or internal Trainer metrics like 'eval_runtime'
        if isinstance(metric_value, (int, float)) and not metric_name.startswith('eval_runtime') and not metric_name.startswith('eval_samples_per_second') and not metric_name.startswith('eval_steps_per_second') and not metric_name.startswith('epoch'):
            if metric_name not in aggregated_metrics:
                aggregated_metrics[metric_name] = []
            aggregated_metrics[metric_name].append(metric_value)

print("\n--- Aggregated Metrics Across All Folds ---")
for metric_name, values in aggregated_metrics.items():
    mean_value = np.mean(values)
    std_value = np.std(values)
    print(f"{metric_name}: Mean = {mean_value:.4f}, Std = {std_value:.4f}")